# rStar-Math Training on Colab A100 80GB
## 24시간 완성 플랜: SFT (15만) + RM (10만) + 평가

**예상 소요 시간**: ~20시간
- 환경 설정: 1시간
- SFT 훈련: 10시간
- RM 훈련: 8시간
- 평가: 1시간

**요구사항**: Colab A100 80GB GPU

## Step 0: GPU 확인 및 Google Drive 마운트

In [ ]:
# GPU 확인
!nvidia-smi

# Google Drive 마운트 (모델 백업용)
from google.colab import drive
drive.mount('/content/drive')

# 작업 디렉토리 생성
!mkdir -p /content/drive/MyDrive/rstar_models
!mkdir -p /content/rStar-Math/data

## Step 1: 환경 설정 (15분)

In [ ]:
# 저장소 클론
!git clone https://github.com/microsoft/rStar-Math.git
%cd rStar-Math

In [ ]:
# 필수 패키지 설치
!pip install -q vllm==0.6.6.post1
!pip install -q transformers==4.45.2
!pip install -q trl==0.9.6
!pip install -q datasets
!pip install -q accelerate
!pip install -q omegaconf termcolor jsonlines pebble matplotlib
!pip install -q scipy word2number func_timeout editdistance
!pip install -q antlr4-python3-runtime==4.11.1

print("✅ Package installation complete!")

In [ ]:
# 평가 툴킷 설치
!git clone https://github.com/MARIO-Math-Reasoning/MARIO_EVAL.git
%cd MARIO_EVAL/latex2sympy
!pip install -q .
%cd ..
!pip install -q -e .
%cd /content/rStar-Math

print("✅ Evaluation toolkit installed!")

In [ ]:
# CUDA 환경 변수 설정 (vllm 호환성)
import os
import site

nvjitlink_path = site.getsitepackages()[0] + '/nvidia/nvjitlink/lib'
os.environ['LD_LIBRARY_PATH'] = nvjitlink_path + ':' + os.environ.get('LD_LIBRARY_PATH', '')

print("✅ CUDA environment configured!")
print(f"LD_LIBRARY_PATH: {os.environ['LD_LIBRARY_PATH']}")

## Step 2: 데이터 다운로드 및 준비 (30-45분)

In [ ]:
from datasets import load_dataset
import json
from tqdm import tqdm

# SFT 데이터 다운로드
print("📥 Downloading SFT dataset...")
sft_dataset = load_dataset("ElonTusk2001/rstar_sft", split="train")
print(f"Total SFT samples: {len(sft_dataset):,}")

# 15만 개 샘플링
sft_sampled = sft_dataset.shuffle(seed=42).select(range(150000))
print(f"Using {len(sft_sampled):,} samples for SFT training")

# JSON 형식으로 저장
print("💾 Saving SFT data...")
sft_data = []
for item in tqdm(sft_sampled, desc="Processing SFT data"):
    sft_data.append({
        "query": item["query"],
        "response": item["response"]
    })

with open("/content/rStar-Math/data/sft_train.json", "w", encoding="utf-8") as f:
    json.dump(sft_data, f, ensure_ascii=False, indent=2)

print("✅ SFT data saved to /content/rStar-Math/data/sft_train.json")
print(f"   File size: {os.path.getsize('/content/rStar-Math/data/sft_train.json') / 1024 / 1024:.2f} MB")

In [ ]:
# PPM 데이터 다운로드
print("\n📥 Downloading PPM dataset...")
ppm_dataset = load_dataset("ElonTusk2001/rstar_ppm", split="train")
print(f"Total PPM samples: {len(ppm_dataset):,}")

# 10만 개 샘플링
ppm_sampled = ppm_dataset.shuffle(seed=42).select(range(100000))
print(f"Using {len(ppm_sampled):,} samples for PPM training")

# JSON 형식으로 저장
print("💾 Saving PPM data...")
ppm_data = []
for item in tqdm(ppm_sampled, desc="Processing PPM data"):
    ppm_data.append({
        "prompt": item["prompt"],
        "pos": item["pos"],
        "neg": item["neg"],
        "pos_count": item["pos_count"],
        "neg_count": item["neg_count"]
    })

with open("/content/rStar-Math/data/ppm_train.json", "w", encoding="utf-8") as f:
    json.dump(ppm_data, f, ensure_ascii=False, indent=2)

print("✅ PPM data saved to /content/rStar-Math/data/ppm_train.json")
print(f"   File size: {os.path.getsize('/content/rStar-Math/data/ppm_train.json') / 1024 / 1024:.2f} MB")

# 메모리 정리
del sft_dataset, sft_sampled, sft_data, ppm_dataset, ppm_sampled, ppm_data
import gc
gc.collect()

print("\n✅ All data preparation complete!")

## Step 3: SFT 모델 훈련 (10시간)

**파라미터 설정**:
- 데이터: 15만 샘플
- Epoch: 1
- Batch size: 2 (per device)
- Gradient accumulation: 8
- Effective batch size: 16
- 예상 시간: ~10시간

In [ ]:
import os

# 환경 변수 설정
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['NCCL_P2P_DISABLE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['FLASH_ATTENTION_DETERMINISTIC'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("🚀 Starting SFT training...")
print("⏱️  Estimated time: 10 hours")
print("📊 Training configuration:")
print("   - Model: Qwen2.5-Math-7B-Instruct")
print("   - Samples: 150,000")
print("   - Epochs: 1")
print("   - Batch size: 2 x 8 (grad_accum) = 16 effective")
print("   - Learning rate: 7e-6")
print("   - Max length: 2048")
print("\n" + "="*60 + "\n")

In [ ]:
%%time

!python3 train/train_SFT.py \
    --model_name_or_path "Qwen/Qwen2.5-Math-7B-Instruct" \
    --data_path "/content/rStar-Math/data/sft_train.json" \
    --data_length 150000 \
    --bf16 True \
    --output_dir "/content/rStar-Math/models/policy_model" \
    --num_train_epochs 1 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 5000 \
    --save_total_limit 2 \
    --learning_rate 7e-6 \
    --weight_decay 0.1 \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 10 \
    --model_max_length 2048 \
    --gradient_checkpointing True \
    --attn_impl sdpa

In [ ]:
# SFT 모델을 Google Drive에 백업
import shutil

print("💾 Backing up SFT model to Google Drive...")
shutil.copytree(
    "/content/rStar-Math/models/policy_model",
    "/content/drive/MyDrive/rstar_models/policy_model",
    dirs_exist_ok=True
)
print("✅ SFT model backup complete!")

## Step 4: SFT 모델 평가 (20분)

In [ ]:
# GSM8K 평가
print("📊 Evaluating SFT model on GSM8K...")
!python eval.py \
    --model "/content/rStar-Math/models/policy_model" \
    --device 0 \
    --task gsm8k

In [ ]:
# GSM8K 결과 확인
!python eval_output.py \
    --file_path "/content/rStar-Math/models/policy_model/gsm8k.jsonl"

In [ ]:
# MATH 평가
print("\n📊 Evaluating SFT model on MATH...")
!python eval.py \
    --model "/content/rStar-Math/models/policy_model" \
    --device 0 \
    --task math

In [ ]:
# MATH 결과 확인
!python eval_output.py \
    --file_path "/content/rStar-Math/models/policy_model/math.jsonl"

## Step 5: Reward Model 훈련 (8시간)

**파라미터 설정**:
- 데이터: 10만 샘플
- Epoch: 1
- Batch size: 4 (per device)
- Gradient accumulation: 8
- Effective batch size: 32
- 예상 시간: ~8시간

In [ ]:
print("🚀 Starting Reward Model training...")
print("⏱️  Estimated time: 8 hours")
print("📊 Training configuration:")
print("   - Base model: SFT policy model")
print("   - Samples: 100,000")
print("   - Epochs: 1")
print("   - Batch size: 4 x 8 (grad_accum) = 32 effective")
print("   - Learning rate: 7e-6")
print("   - Max length: 2048")
print("\n" + "="*60 + "\n")

In [ ]:
%%time

!python train/train_RM.py \
    --model_name_or_path="/content/rStar-Math/models/policy_model" \
    --output_dir="/content/rStar-Math/models/reward_model_ckpt" \
    --pair_json_path "/content/rStar-Math/data/ppm_train.json" \
    --per_device_train_batch_size=4 \
    --per_device_eval_batch_size=4 \
    --num_train_epochs=1 \
    --gradient_accumulation_steps=8 \
    --gradient_checkpointing=True \
    --learning_rate=7e-6 \
    --remove_unused_columns=False \
    --optim="adamw_torch" \
    --logging_steps=10 \
    --eval_strategy="steps" \
    --eval_steps=2500 \
    --save_steps=2500 \
    --load_best_model_at_end=True \
    --save_total_limit=2 \
    --max_length=2048 \
    --bf16=True \
    --attn_impl eager

In [ ]:
# Reward Model 포맷 변환
print("🔄 Converting Reward Model format...")
!python train/save_rm.py \
    --sft_model_path "/content/rStar-Math/models/policy_model" \
    --rm_ckpt_path "/content/rStar-Math/models/reward_model_ckpt" \
    --rm_save_path "/content/rStar-Math/models/reward_model"

print("✅ Reward Model conversion complete!")

In [ ]:
# Reward Model을 Google Drive에 백업
import shutil

print("💾 Backing up Reward Model to Google Drive...")
shutil.copytree(
    "/content/rStar-Math/models/reward_model",
    "/content/drive/MyDrive/rstar_models/reward_model",
    dirs_exist_ok=True
)
print("✅ Reward Model backup complete!")

## Step 6: MCTS 추론 평가 (1시간)

Policy Model과 Reward Model을 사용한 MCTS 추론으로 최종 성능 평가

In [ ]:
# AIME 2024 평가 (작은 데이터셋으로 빠른 테스트)
print("🌲 Running MCTS inference on AIME 2024...")
print("⏱️  Estimated time: ~30 minutes")

!CUDA_VISIBLE_DEVICES=0 python main.py \
    --qaf eval_data/aime2024_test.json \
    --custom_cfg config/sft_eval_mcts.yaml \
    --model_dir "/content/rStar-Math/models/policy_model" \
    --reward_model_dir "/content/rStar-Math/models/reward_model"

In [ ]:
# AMC 2023 평가
print("\n🌲 Running MCTS inference on AMC 2023...")
print("⏱️  Estimated time: ~30 minutes")

!CUDA_VISIBLE_DEVICES=0 python main.py \
    --qaf eval_data/amc23_test.json \
    --custom_cfg config/sft_eval_mcts.yaml \
    --model_dir "/content/rStar-Math/models/policy_model" \
    --reward_model_dir "/content/rStar-Math/models/reward_model"

## Step 7: 결과 요약 및 백업

In [ ]:
# 결과 파일 확인
import os
import json

print("📊 Training and Evaluation Summary")
print("="*60)

# SFT 평가 결과
print("\n🎯 SFT Model (Greedy Decoding):")
if os.path.exists("/content/rStar-Math/models/policy_model/gsm8k.jsonl"):
    print("   ✅ GSM8K results available")
if os.path.exists("/content/rStar-Math/models/policy_model/math.jsonl"):
    print("   ✅ MATH results available")

# MCTS 평가 결과 확인
print("\n🌲 MCTS Inference Results:")
mcts_results = []
for root, dirs, files in os.walk("/content/rStar-Math"):
    for file in files:
        if "aime2024" in file and file.endswith(".json"):
            print(f"   ✅ AIME 2024: {os.path.join(root, file)}")
            mcts_results.append(os.path.join(root, file))
        if "amc23" in file and file.endswith(".json"):
            print(f"   ✅ AMC 2023: {os.path.join(root, file)}")
            mcts_results.append(os.path.join(root, file))

print("\n💾 Saved Models:")
print("   📁 Policy Model: /content/rStar-Math/models/policy_model")
print("   📁 Reward Model: /content/rStar-Math/models/reward_model")
print("   ☁️  Backup: /content/drive/MyDrive/rstar_models/")

print("\n" + "="*60)

In [ ]:
# 모든 결과를 Google Drive에 백업
import shutil
import os

print("💾 Final backup to Google Drive...")

# 결과 디렉토리 생성
results_dir = "/content/drive/MyDrive/rstar_models/results"
os.makedirs(results_dir, exist_ok=True)

# SFT 평가 결과 백업
for filename in ["gsm8k.jsonl", "math.jsonl"]:
    src = f"/content/rStar-Math/models/policy_model/{filename}"
    if os.path.exists(src):
        shutil.copy(src, f"{results_dir}/sft_{filename}")
        print(f"   ✅ Backed up {filename}")

# MCTS 결과 백업
for result_file in mcts_results:
    if os.path.exists(result_file):
        basename = os.path.basename(result_file)
        shutil.copy(result_file, f"{results_dir}/mcts_{basename}")
        print(f"   ✅ Backed up {basename}")

print("\n✅ All backups complete!")
print(f"📁 Results saved to: {results_dir}")

## 완료! 🎉

### 훈련된 모델:
1. **Policy Model (SFT)**: `/content/drive/MyDrive/rstar_models/policy_model`
2. **Reward Model**: `/content/drive/MyDrive/rstar_models/reward_model`

### 평가 결과:
- SFT Greedy: GSM8K, MATH
- MCTS: AIME 2024, AMC 2023

### 예상 성능:
- **GSM8K**: 70-75% (SFT) → 75-80% (MCTS)
- **MATH**: 35-40% (SFT) → 40-45% (MCTS)
- **AIME 2024**: 10-15% (MCTS)

### 다음 단계:
1. 더 많은 데이터로 재훈련 (전체 119만 SFT, 141만 PPM)
2. 더 많은 epoch 훈련 (2-3 epochs)
3. 더 큰 모델 시도 (Qwen2.5-Math-14B)
4. MCTS 파라미터 튜닝 (iterations, n_generate_sample)